# Get to Know a Dataset: TAIR Multimodal Imaging Dataset for Cerebrovascular Disease

**AWS Open Data Sponsorship Program tutorial**
**Audience:** medical imaging researchers, engineers, and students
**Duration:** 30–45 minutes

This hands-on notebook shows how to discover the public TAIR S3 bucket, use its `manifest.csv` to find a patient, download just that patient's data, and run a few simple operations on a CTA/MRA volume (NIfTI) and an X-ray fluoroscopy video (MP4). It ends with a reproducible community challenge.

No AWS account or credentials are required. Only a few tens of MB are downloaded by default.

**Research-use notice:** Imaging data are not for clinical diagnosis. Respect the dataset license, patient privacy, and institutional requirements.

## 1. Dataset overview and research questions

TAIR is a multimodal cerebrovascular imaging dataset covering the imaging modalities used in neurovascular procedures. It was collected as part of routine critical care across multiple centers and released to support the TAIR (Toolkit for Autonomous Interventions and Robotics) testbed and community-wide clinical and diagnostic research.

| Component | Contents | Format |
|---|---|---|
| **CTA / MRA** | Computed Tomography Angiography and Magnetic Resonance Angiography of the head (`-H`), neck (`-N`), or head and neck (`-HN`). The current release has 500 studies; the final release (12/30/2027) is planned at 2,000. | NIfTI (`.nii.gz`) |
| **X-ray fluoroscopy** | Videos from interventional procedures, paired with CTA/MRA from the same patient where available. The current release has 150 datasets; the final release is planned at 1,250. | MP4, plus per-frame OCR CSV |
| **Anatomical models** *(coming soon)* | Segmented, centerlined, and annotated models, plus statistical shape models | VTK · NIfTI · OpenUSD · JSON graphs |

**Example research questions**
- How can researchers access one patient's studies without downloading the full 70 GB archive?
- What volume dimensions, voxel spacings, and intensity ranges do the CTA/MRA scans have?
- How are fluoroscopy runs organized, and what does the per-frame overlay metadata (frame type, projection angle, timing) contain?
- How might paired CTA/MRA and fluoroscopy support vessel visualization, segmentation, 2D/3D registration, or multimodal research?

**Publication details**

| Field | Value |
|---|---|
| Dataset homepage | https://tair.kitware.com/clinical-data/ |
| AWS Registry entry | *Pending — Registry listing not yet published* |
| S3 URI | `s3://multimodal-imaging-for-cerebrovascular-disease/` (region `us-east-1`) |
| Version / release date | v1.0, November 2026 |
| License | [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) |
| Citation | See [Section 11](#11.-Where-to-go-next) (BibTeX) |
| Dataset size / cohort | 70 GB; 495 patients in `manifest.csv` (500 CTA/MRA studies and 150 fluoroscopy datasets in the current release) |
| De-identification | PHI was removed from image metadata before release. Defacing algorithms removed identifying facial structures from CTA/MRA, and burned-in text and identifying structures were removed from fluoroscopy. See the [dataset page](https://tair.kitware.com/clinical-data/). |
| Contact | [TAIRcontact@kitware.com](mailto:TAIRcontact@kitware.com) |

## 2. Environment and costs

Python 3.10+ recommended. This tutorial uses `boto3`/`botocore` for anonymous S3 access, `pandas` for the manifest and OCR tables, `nibabel` to read NIfTI volumes, `imageio` with the PyAV plugin to decode MP4 video, and `numpy`/`matplotlib`.

The bucket is public and can be read anonymously. Your compute, notebook runtime, and some network transfers may still incur costs. This notebook bounds listings and refuses downloads over a size limit by default.

Install dependencies in your notebook environment if necessary:

In [ ]:
# Uncomment and run if dependencies are missing:
# %pip install boto3 botocore pandas numpy matplotlib nibabel "imageio[pyav]"

In [ ]:
from pathlib import Path
import os
import tempfile

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import imageio.v3 as iio
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

print('Ready: boto3 / pandas / numpy / matplotlib / nibabel / imageio')

## 3. Configure the public dataset location

The dataset lives in a public bucket in `us-east-1`. Anonymous (unsigned) access is intentional, so no AWS credentials are required.

In [ ]:
BUCKET = os.environ.get('TAIR_S3_BUCKET', 'multimodal-imaging-for-cerebrovascular-disease')
REGION = 'us-east-1'
MAX_DOWNLOAD_MB = 100  # refuse to download any single object larger than this
DATA_DIR = Path(os.environ.get('TAIR_DATA_DIR', Path(tempfile.gettempdir()) / 'tair_tutorial'))
DATA_DIR.mkdir(parents=True, exist_ok=True)

s3 = boto3.client('s3', region_name=REGION,
                  config=Config(signature_version=UNSIGNED, retries={'max_attempts': 3}))


def download(key, dest_dir=DATA_DIR, max_mb=MAX_DOWNLOAD_MB):
    # Download one object with a size guard. Returns the local path, or None if skipped.
    size = s3.head_object(Bucket=BUCKET, Key=key)['ContentLength']
    if size > max_mb * 1024**2:
        print(f'Skipped {key}: {size/1024**2:.1f} MiB is over the {max_mb} MiB limit.')
        return None
    dest = Path(dest_dir) / key
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists() or dest.stat().st_size != size:
        s3.download_file(BUCKET, key, str(dest))
    print(f'{key}  ({size/1024**2:.2f} MiB) -> {dest}')
    return dest


print('Bucket:', f's3://{BUCKET}/')
print('Local cache:', DATA_DIR)

## 4. Discover the bucket layout

A delimited listing of the bucket root shows the top-level files and one prefix per patient. It doesn't enumerate the full dataset.

```
s3://multimodal-imaging-for-cerebrovascular-disease/
├── manifest.csv                     # one row per patient: what data is available
├── tair-project-data.zip            # the full dataset as a single ~70 GB archive
├── tair-project-data.zip.sha256     # checksum for the archive
└── <patient_id>/                    # e.g. 00008/ (video file stems vary, e.g. A1_t00 or B1_t01)
    ├── SCANS/
    │   └── {CTA,MRA}-{H,N,HN}_defaced.nii.gz
    └── FLUORO/
        └── <run>/                   # A, B, C, ... one folder per fluoroscopy run
            ├── A1_t00_masked.mp4             # full de-identified video
            ├── A1_t00_masked_ocr.csv         # per-frame overlay text (OCR)
            ├── A1_t00_masked_roadmap.mp4     # only frames whose frame_type is "Roadmap"
            ├── A1_t00_masked_WC<n>.mp4       # only frames whose frame_type is "WC<n>"
            └── A1_t00_masked_unlabeled.mp4   # frames with no recognized frame type
```

In [ ]:
paginator = s3.get_paginator('list_objects_v2')
patient_prefixes, root_files = [], []
for page in paginator.paginate(Bucket=BUCKET, Delimiter='/'):
    patient_prefixes += [p['Prefix'] for p in page.get('CommonPrefixes', [])]
    root_files += page.get('Contents', [])

for item in root_files:
    print(f"{item['Size']/1024**2:12.2f} MiB  {item['Key']}")
print(f'\n{len(patient_prefixes)} patient prefixes, e.g. {patient_prefixes[:5]}')
bucket_patients = {p.rstrip('/') for p in patient_prefixes}

> **Need everything?** Download the whole dataset as one archive with the AWS CLI. Make sure you have enough disk space.
> ```bash
> aws s3 cp s3://multimodal-imaging-for-cerebrovascular-disease/tair-project-data.zip . --no-sign-request
> aws s3 cp s3://multimodal-imaging-for-cerebrovascular-disease/tair-project-data.zip.sha256 . --no-sign-request
> sha256sum -c tair-project-data.zip.sha256
> ```
> The rest of this notebook downloads individual objects instead.

## 5. Explore the manifest

`manifest.csv` has one row per patient:

| Column | Meaning |
|---|---|
| `patient_id` | Zero-padded ID; also the S3 prefix (keep it as a string) |
| `fluoro` | `yes` if the patient has fluoroscopy data |
| `fluoro_A`, `fluoro_B`, `fluoro_C` | Number of fluoroscopy runs in `FLUORO/A`, `FLUORO/B`, `FLUORO/C` |
| `scans` | `yes` if the patient has CTA/MRA data |
| `CTA-H`, `CTA-N`, `CTA-HN`, `MRA-H`, `MRA-N`, `MRA-HN` | Number of scans of each type (Head, Neck, Head and Neck) |
| `notes` | Free-text notes, e.g. when a patient has more than one scan of a type |

In [ ]:
manifest_path = download('manifest.csv')
manifest = pd.read_csv(manifest_path, dtype={'patient_id': str})
manifest.head()

In [ ]:
SCAN_TYPES = ['CTA-H', 'CTA-N', 'CTA-HN', 'MRA-H', 'MRA-N', 'MRA-HN']
FLUORO_RUNS = ['fluoro_A', 'fluoro_B', 'fluoro_C']

has_fluoro = manifest['fluoro'].eq('yes')
has_scans = manifest['scans'].eq('yes')
print(f'Patients:                 {len(manifest)}')
print(f'  with CTA/MRA:           {has_scans.sum()}')
print(f'  with fluoroscopy:       {has_fluoro.sum()}')
print(f'  with both (paired):     {(has_fluoro & has_scans).sum()}')
missing = sorted(bucket_patients - set(manifest['patient_id']))
if missing:
    print(f'Patient prefixes in the bucket but not in the manifest: {missing}')

counts = manifest[SCAN_TYPES + FLUORO_RUNS].sum()
ax = counts.plot.bar(figsize=(8, 3), color=['tab:blue'] * len(SCAN_TYPES) + ['tab:orange'] * len(FLUORO_RUNS))
ax.set_ylabel('count')
ax.set_title('Scans and fluoroscopy runs listed in manifest.csv')
plt.tight_layout()
plt.show()

notes = manifest.dropna(subset=['notes'])
if len(notes):
    print('Rows with notes:')
    display(notes[['patient_id', 'notes']])

## 6. Pick a patient and list their files

Paired patients, who have both CTA/MRA and fluoroscopy, are the most interesting for multimodal work. By default this cell picks the first paired patient. Set `TAIR_PATIENT_ID` (or edit `PATIENT_ID`) to choose another.

The manifest tells you *what* a patient has. Listing the patient's prefix gives the exact object keys and sizes, so treat the listing as the source of truth for file names.

In [ ]:
paired = manifest[has_fluoro & has_scans]
PATIENT_ID = os.environ.get('TAIR_PATIENT_ID', paired['patient_id'].iloc[0])

row = manifest.set_index('patient_id').loc[PATIENT_ID]
print(f'Patient {PATIENT_ID}:')
print('  scans: ', {t: int(row[t]) for t in SCAN_TYPES if row[t]})
print('  fluoro:', {r: int(row[r]) for r in FLUORO_RUNS if row[r]})

patient_objects = [o for page in paginator.paginate(Bucket=BUCKET, Prefix=f'{PATIENT_ID}/')
                   for o in page.get('Contents', [])]
files = pd.DataFrame({'key': [o['Key'] for o in patient_objects],
                      'size_mib': [round(o['Size'] / 1024**2, 2) for o in patient_objects]})
files

## 7. CTA/MRA: load a NIfTI volume

The volumes are defaced, gzipped NIfTI files. `nibabel` gives the voxel array, the voxel spacing, and the affine that maps voxel indices to scanner coordinates. NIfTI headers carry no patient identifiers.

In [ ]:
scan_keys = sorted(k for k in files['key'] if '/SCANS/' in k and k.endswith('.nii.gz'))
# Prefer a CTA for the intensity steps below (CT values are in Hounsfield units)
scan_key = next((k for k in scan_keys if '/CTA-' in k), scan_keys[0] if scan_keys else None)
scan_path = download(scan_key) if scan_key else None
is_ct = scan_key is not None and '/CTA-' in scan_key

vol = None
if scan_path:
    # Reorient to the closest canonical (RAS+) orientation so the axes are predictable
    img = nib.as_closest_canonical(nib.load(scan_path))
    vol = img.get_fdata(dtype=np.float32)
    spacing = img.header.get_zooms()[:3]
    print('Shape (voxels):    ', vol.shape)
    print('Spacing (mm):      ', tuple(round(float(s), 3) for s in spacing))
    print('Orientation:       ', ''.join(nib.aff2axcodes(img.affine)), '(after reorienting)')
    print('Field of view (mm):', tuple(round(n * float(s), 1) for n, s in zip(vol.shape, spacing)))
    print('Intensity range:   ', vol.min(), 'to', vol.max(), '(HU)' if is_ct else '(arbitrary MR units)')
else:
    print('This patient has no CTA/MRA volume; pick another PATIENT_ID.')

### A simple operation: maximum intensity projections and a threshold

A **maximum intensity projection (MIP)** keeps the brightest voxel along each ray. On contrast-enhanced CTA this makes vessels stand out, so MIPs are a common first look at vascular anatomy. Through a whole head, though, every ray hits the skull and the projection shows only bone. The usual fix is a **thin-slab MIP** that projects only a slab a few centimeters thick. The cells below:

1. compute axial, coronal, and sagittal slab MIPs through the center of the volume and display them with physically correct aspect ratios, and
2. count voxels in a contrast-enhanced range (roughly 150–600 HU for CTA) to estimate how much of the volume they occupy. This range also catches partial-volume voxels at the edge of the skull and calcifications, so it's a toy measurement, **not** a vessel segmentation.

The display windowing is for visualization only and is not a clinically calibrated display pipeline.

In [ ]:
if vol is not None:
    sx, sy, sz = (float(s) for s in spacing)
    if is_ct:
        vmin, vmax = 0, 600  # HU window that emphasizes contrast-filled vessels
    else:
        vmin, vmax = np.percentile(vol, [50, 99.9])

    # In RAS+ voxel order, axis 0 runs left->right, axis 1 posterior->anterior, and axis 2
    # inferior->superior. With origin='lower', anterior/superior are at the top of each view
    # and the patient's right is on the screen's right (neurological convention).
    SLAB_MM = 20  # slab thickness; try larger values, or the whole volume, to see the skull take over

    def slab_mip(axis):
        half = max(1, round(SLAB_MM / 2 / float(spacing[axis])))
        center = vol.shape[axis] // 2
        index = [slice(None)] * 3
        index[axis] = slice(max(0, center - half), center + half)
        return vol[tuple(index)].max(axis=axis).T

    views = [
        ('Axial',    slab_mip(2), sy / sx),
        ('Coronal',  slab_mip(1), sz / sx),
        ('Sagittal', slab_mip(0), sz / sy),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, (title, mip, aspect) in zip(axes, views):
        ax.imshow(mip, cmap='gray', vmin=vmin, vmax=vmax, origin='lower', aspect=aspect)
        ax.set_title(f'{title} {SLAB_MM} mm slab MIP')
        ax.axis('off')
    fig.suptitle(f'Patient {PATIENT_ID} · {Path(scan_key).name} · research visualization only')
    plt.tight_layout()
    plt.show()

In [ ]:
if vol is not None and is_ct:
    lo, hi = 150, 600
    mask = (vol >= lo) & (vol <= hi)
    voxel_ml = sx * sy * sz / 1000.0
    print(f'Voxels in [{lo}, {hi}] HU: {mask.sum():,} of {mask.size:,} ({100 * mask.mean():.2f}%)')
    print(f'Approximate volume:     {mask.sum() * voxel_ml:.1f} mL')

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(vol[vol > -1000].ravel(), bins=200, range=(-1000, 2000), color='gray')
    ax.axvspan(lo, hi, color='tab:red', alpha=0.2, label=f'{lo}–{hi} HU')
    ax.set_yscale('log')
    ax.set_xlabel('Hounsfield units')
    ax.set_ylabel('voxels (log)')
    ax.legend()
    ax.set_title('CTA intensity histogram (air excluded)')
    plt.tight_layout()
    plt.show()
elif vol is not None:
    print('Selected scan is MRA: intensities are not in HU, so the HU threshold is skipped.')

## 8. Fluoroscopy: overlay metadata and video

Each fluoroscopy run folder holds the full de-identified video and an OCR table with **one row per video frame**. The OCR columns are text read from the original on-screen overlays, such as:

- `frame_type`: the label used to split the video into the `_roadmap`, `_WC<n>`, and `_unlabeled` MP4s
- `relative_time`, `time_known`, `time_origin_uncertain`: elapsed time within the run and how reliable it is
- `BottomLeft_*` / `BottomRight_*`: raw overlay fields; for example, `BottomLeft_3` usually holds the C-arm projection angle (e.g. `LAO 25°/ CAUD 15°`)

Overlay contents vary by site and system, so check the values before relying on any column.

In [ ]:
fluoro_videos = sorted(k for k in files['key']
                       if '/FLUORO/' in k and k.endswith('_masked.mp4'))
video_key = fluoro_videos[0] if fluoro_videos else None
ocr_key = video_key.replace('_masked.mp4', '_masked_ocr.csv') if video_key else None

ocr = None
if ocr_key and ocr_key in set(files['key']):
    ocr = pd.read_csv(download(ocr_key))
    print(f'{len(ocr)} OCR rows (one per frame)\n')
    print(ocr['frame_type'].value_counts().to_string(), '\n')
    print('Projection angles (BottomLeft_3):')
    print(ocr['BottomLeft_3'].value_counts().head().to_string())
    display(ocr.head())
else:
    print('This patient has no fluoroscopy run; pick another PATIENT_ID.')

### A simple operation: per-frame brightness over time

The MP4s are 8-bit RGB with identical channels, so one channel is enough. Instead of loading the whole video into memory, which can be several GB for long runs, the next cell streams frames one at a time. It records each frame's mean intensity and keeps one frame per `frame_type` to display. Plotting the brightness trace colored by `frame_type` shows how the run is segmented.

In [ ]:
video_path = download(video_key) if video_key else None

if video_path:
    meta = iio.immeta(video_path, plugin='pyav')
    print(f"fps: {meta.get('fps')}, duration: {meta.get('duration', 0):.1f} s")

    frame_types = ocr['frame_type'].fillna('Unlabeled').to_numpy() if ocr is not None else None
    means, examples = [], {}
    for i, frame in enumerate(iio.imiter(video_path, plugin='pyav')):
        gray = frame[..., 0]
        means.append(gray.mean())
        label = frame_types[i] if frame_types is not None and i < len(frame_types) else 'video'
        examples.setdefault(label, gray)  # first frame of each type
    means = np.asarray(means)
    print(f'Decoded {len(means)} frames of {gray.shape[1]}x{gray.shape[0]} pixels')
    if ocr is not None and len(ocr) != len(means):
        print(f'Note: OCR rows ({len(ocr)}) and frames ({len(means)}) differ; plotting the overlap.')

In [ ]:
if video_path:
    fps = meta.get('fps') or 30
    t = np.arange(len(means)) / fps

    fig, ax = plt.subplots(figsize=(12, 3))
    if frame_types is not None:
        n = min(len(means), len(frame_types))
        for label in pd.unique(frame_types[:n]):
            sel = frame_types[:n] == label
            ax.scatter(t[:n][sel], means[:n][sel], s=2, label=label)
        ax.legend(markerscale=5, title='frame_type')
    else:
        ax.plot(t, means)
    ax.set_xlabel('time (s, from video frame rate)')
    ax.set_ylabel('mean pixel intensity')
    ax.set_title(f'Patient {PATIENT_ID} · {Path(video_key).relative_to(PATIENT_ID)}')
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 4), squeeze=False)
    for ax, (label, img) in zip(axes[0], examples.items()):
        ax.imshow(img, cmap='gray')
        ax.set_title(f'first "{label}" frame')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 9. Reproducibility

Record the **dataset version**, the S3 object keys you used, software versions, and analysis settings. Don't redistribute data outside the license terms, and don't infer clinical outcomes from a single example.

In [ ]:
import importlib.metadata as metadata
for package in ['boto3', 'pandas', 'numpy', 'matplotlib', 'nibabel', 'imageio', 'av']:
    try:
        print(f'{package:12s}', metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f'{package:12s} not installed')
print('\nPatient:', PATIENT_ID)
for key in [k for k in (scan_key, video_key, ocr_key) if k]:
    print(f'  s3://{BUCKET}/{key}')

## 10. Community challenge: reproducible cerebrovascular visualization

**Goal:** Using a documented public TAIR subset, create a reproducible notebook that characterizes the image data and produces a meaningful visualization or baseline analysis.

**Suggested tasks**
1. Use `manifest.csv` to select a small, well-defined subset (for example, 10 paired patients with `CTA-H` and a fluoroscopy run `A`) and document the selection criteria.
2. Summarize volume dimensions, voxel spacing, and intensity ranges across the subset, plus fluoroscopy run lengths, frame types, and projection angles from the OCR tables.
3. Create at least one visualization with stated windowing/normalization, such as a CTA MIP next to a fluoroscopy frame acquired at a similar projection angle.
4. *(Optional)* Develop a simple vessel-enhancement or segmentation baseline, or a 2D/3D alignment between CTA and fluoroscopy, and document its limitations. Don't present it as clinically validated.
5. Report dataset version, code dependencies, computational cost, and the citation below.

**Success criteria:** another researcher can reproduce the result from the public bucket without private credentials or patient identifiers.

**Responsible use:** evaluate demographic and acquisition bias, don't attempt patient re-identification, and follow the CC BY 4.0 license and your institutional review requirements.

## 11. Where to go next

- Official dataset documentation: https://tair.kitware.com/clinical-data/
- License: [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)
- Contact: [TAIRcontact@kitware.com](mailto:TAIRcontact@kitware.com)
- AWS Registry listing: *pending publication*
- AWS Open Data Sponsorship Program: https://aws.amazon.com/opendata/open-data-sponsorship-program/
- AWS tutorial onboarding guidance: https://assets.opendata.aws/aws-onboarding-handbook-for-data-providers.pdf
- AWS example notebook template: https://github.com/awslabs/open-data-docs/tree/main/get-to-know-a-dataset

### Citation

```bibtex
@dataset{tair2026,
  author    = {Lee, Yueh Z. and de Siqueira Bucci, Ana C. and Ho, James and Paniagua, Beatriz and Vicory, Jared and Ravikumar, Sadhana and Nagler, Mike and Webb, Jeffrey B. and Clipp, Rachel B.},
  title     = {{TAIR}: Multimodal Imaging Dataset
              for Cerebrovascular Disease},
  year      = {2026},
  version   = {1.0},
  publisher = {AWS Registry of Open Data},
  url       = {https://tair.kitware.com/clinical-data},
}
```

Contributors: Kitware, Inc. and the University of North Carolina at Chapel Hill.